# Day 01 — How text becomes numbers

A neural network cannot multiply the word `cat`. It multiplies floats. So somewhere between your
keyboard and the first matrix multiply, text has to become numbers — and *how* that conversion
happens explains a surprising amount of real model behaviour: why models are bad at arithmetic, why
they miscount letters, why Hindi costs four times more than English, and why `"strawberry"` is a
famous failure case.

**Time budget: 60 minutes.** Each segment carries its own timebox.

## How to use this notebook

1. Read the markdown, then run the cell below it before reading on. Cells build on each other, so
   run them in order.
2. Where a cell says **predict first**, actually write down your guess. Being wrong is the part
   that sticks.
3. Finish with the exercises at the bottom. Solutions are included, but try first.

## Agenda

| # | Segment | Time |
| - | ------- | ---- |
| 0 | Why numbers at all — the `ord()` baseline and its emptiness | 3 min |
| 1 | One-hot encoding, and watching it fail numerically | 8 min |
| 2 | Tokenization for real, hands-on with `tiktoken` | 12 min |
| 3 | Implement byte-pair encoding from scratch | 10 min |
| 4 | Token IDs to vectors: the embedding matrix | 5 min |
| 5 | Word2Vec from scratch: skip-gram, negative sampling, training | 15 min |
| 6 | Why Word2Vec is not enough: context and order | 7 min |
| 7 | Exercises and self-check quiz | — |

## The pipeline we are building

Everything in this hour is one of these arrows:

```
"the cat sat"  ->  [1820, 8415, 7731]  ->  [[0.21, -0.04, ...], [...], [...]]  ->  model
     text            token IDs (ints)              dense vectors (floats)
              segments 2-3                  segments 4-5
```

In [2]:
import numpy as np

np.random.seed(0)
np.set_printoptions(precision=3, suppress=True, linewidth=100)


def cosine(a, b):
    # 1.0 = same direction, 0.0 = unrelated, -1.0 = opposite
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


print("numpy", np.__version__, "- ready")

numpy 2.5.3 - ready


---
## 0. Why numbers at all? (3 min)

The problem in one sentence: **a model is a pile of matrix multiplications, and you cannot multiply
a string.** So we need a function from text to numbers.

The laziest such function already exists in Python. Every character has a code point.

In [3]:
word = "cat"
codes = [ord(ch) for ch in word]

print(f"{word!r} -> {codes}")
print("and back again ->", "".join(chr(c) for c in codes))

'cat' -> [99, 97, 116]
and back again -> cat


That works: it is reversible, and nothing is lost. So why isn't this how models read text?

Because **we do not just want numbers, we want numbers where distance means something.** Watch what
this encoding thinks is similar. Predict first: which pair below should come out closest —
`(cat, car)` or `(cat, dog)`?

In [4]:
def char_vector(w):
    return np.array([ord(c) for c in w], dtype=float)


for a, b in [("cat", "car"), ("cat", "cot"), ("cat", "dog")]:
    d = np.linalg.norm(char_vector(a) - char_vector(b))
    print(f"distance({a}, {b}) = {d:6.2f}")

distance(cat, car) =   2.00
distance(cat, cot) =  14.00
distance(cat, dog) =  19.13


Exactly backwards. Character codes say `cat` is almost the same thing as `car` (distance 2) and that
`dog` is in a different universe (distance 19). Semantically it is the reverse: cats and dogs are
both pets, while a car is a machine.

The lesson that drives the whole rest of the hour:

> Distance in `ord()` space measures **spelling**. We want a space where distance measures
> **meaning**.

Two separate problems are hiding in here, and the rest of the notebook takes them in turn:

1. **How do we chop text into pieces at all?** Characters, words, or something else? → segments 2-3
2. **How do we assign each piece a vector whose geometry encodes meaning?** → segments 4-5

---
## 1. One-hot encoding, and why it fails (8 min)

The standard first answer for turning categories into vectors is **one-hot encoding**: give each
word an index, then represent it as a vector that is all zeros except a single 1 at that index.

No word is arbitrarily "bigger" than another any more, which fixes the obvious flaw in the `ord()`
approach. Let's build it and then look for the non-obvious flaw.

In [5]:
words = ["king", "queen", "man", "woman", "banana"]
vocab = {w: i for i, w in enumerate(words)}
V = len(vocab)


def one_hot(word):
    v = np.zeros(V)
    v[vocab[word]] = 1.0
    return v


for w in words:
    print(f"{w:>7}  id={vocab[w]}  {one_hot(w)}")

   king  id=0  [1. 0. 0. 0. 0.]
  queen  id=1  [0. 1. 0. 0. 0.]
    man  id=2  [0. 0. 1. 0. 0.]
  woman  id=3  [0. 0. 0. 1. 0.]
 banana  id=4  [0. 0. 0. 0. 1.]


Now the test that matters. **Predict first:** how similar will one-hot `king` and `queen` be,
compared to one-hot `king` and `banana`?

In [ ]:
header = "".join(f"{w:>9}" for w in words)
print(f"{'':>7}{header}")
for a in words:
    row = "".join(f"{cosine(one_hot(a), one_hot(b)):>9.2f}" for b in words)
    print(f"{a:>7}{row}")

In [ ]:
print("cosine(king, queen)  =", cosine(one_hot("king"), one_hot("queen")))
print("cosine(king, banana) =", cosine(one_hot("king"), one_hot("banana")))
print()
print("Every off-diagonal entry is exactly 0.0. Not approximately -- exactly.")

That identity matrix is the whole problem. One-hot vectors are **mutually orthogonal by
construction**, so the similarity between any two different words is exactly zero. In this
representation, `king` is precisely as related to `queen` as it is to `banana`.

The model would have to learn every fact about every word from scratch, with no ability to
generalise from `dog` to `puppy`. And there is a second, more practical problem: the vectors are as
wide as the vocabulary.

In [ ]:
V_real = 100_277   # cl100k_base -- the GPT-3.5 / GPT-4 vocabulary
d_model = 768      # the embedding width of a small transformer
seq_len = 1_000    # a modest prompt

onehot_bytes = V_real * seq_len * 4     # float32
dense_bytes = seq_len * d_model * 4

print(f"one-hot: {V_real:>7,} dims x {seq_len:,} tokens x 4 bytes = {onehot_bytes/1e6:8.1f} MB")
print(f"dense:   {d_model:>7,} dims x {seq_len:,} tokens x 4 bytes = {dense_bytes/1e6:8.1f} MB")
print()
print(f"one-hot is {onehot_bytes/dense_bytes:.0f}x bigger, and {100*(1 - 1/V_real):.4f}% of it is zeros")

So one-hot encoding fails on three counts:

| Problem | Consequence |
| ------- | ----------- |
| All distinct words are orthogonal | No notion of similarity, so no generalisation between related words |
| Dimensionality equals vocabulary size | 401 MB for a 1,000-token prompt, 99.999% of it zeros |
| One slot per known word | A word not in the vocabulary cannot be represented at all |

Keep one-hot in mind though — it is not useless. In segment 4 we will see that the embedding lookup
every real model performs *is* a one-hot matrix multiply, just never computed that way.

That third problem is the one we attack next.

---
## 2. Tokenization for real (12 min)

If one slot per word is the problem, why not just use a big word list? Because word-level
vocabularies break in at least five ways:

1. **Out-of-vocabulary words.** Any word you did not see in training is unrepresentable.
2. **Vocabulary explosion.** English has millions of word forms once you count names, typos, URLs
   and product codes. Every one costs an embedding row.
3. **Morphology is thrown away.** `run`, `runs`, `running`, `ran` become four unrelated symbols
   sharing nothing.
4. **Not every language uses spaces.** Chinese, Japanese and Thai do not delimit words.
5. **Typos become total losses.** `teh` is as unknown as a word in a language you never saw.

Watch problem 1 happen.

In [ ]:
train = "the cat sat on the mat".split()
word_vocab = {w: i for i, w in enumerate(dict.fromkeys(train))}
word_vocab["<UNK>"] = len(word_vocab)
inverse = {i: w for w, i in word_vocab.items()}


def word_encode(text):
    return [word_vocab.get(w, word_vocab["<UNK>"]) for w in text.split()]


test = "the dog sat on the sofa"
ids = word_encode(test)

print("input :", test)
print("ids   :", ids)
print("model sees:", " ".join(inverse[i] for i in ids))

`dog` and `sofa` collapse into the same symbol. The model cannot tell an animal from furniture,
because at the level it reads, they are literally the same integer.

### Subword tokenization

The fix used by every modern LLM is to sit **between** characters and words. Common words get their
own token; rare words get split into familiar fragments. The dominant algorithm is **byte-pair
encoding (BPE)**, and OpenAI's implementation is `tiktoken`.

The first call to each encoding downloads its vocabulary file, so this cell needs the network once.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")   # the GPT-3.5 / GPT-4 tokenizer
print("vocabulary size:", f"{enc.n_vocab:,}")

text = "Tokenization is sneaky."
ids = enc.encode(text)

print("\nids       :", ids)
print("round trip:", repr(enc.decode(ids)))
print("\nid       token")
for i in ids:
    print(f"{i:>7}   {enc.decode_single_token_bytes(i)!r}")

Note what came back: not words, but byte strings, some with a **leading space baked in**
(`b' is'`). The space is part of the token. That single design choice causes a lot of surprising
behaviour, which the next few cells explore.

Here is a helper we will reuse.

In [ ]:
def show(text, encoding="cl100k_base"):
    e = tiktoken.get_encoding(encoding)
    ids = e.encode(text)
    pieces = [e.decode_single_token_bytes(i).decode("utf-8", "replace") for i in ids]
    print(f"{len(ids):>3} tokens  {repr(text):<32} {pieces}")


print("--- leading space and capitalisation change everything ---")
for t in ["hello", " hello", "Hello", "HELLO", "hello world"]:
    show(t)

`hello`, ` hello` and `Hello` are three *different* tokens with three different IDs — the model has
to learn separately that they mean the same thing. And `HELLO` does not even fit in one token; it
becomes `HEL` + `LO`.

This is why prompts are sensitive to trailing spaces, and why a stray space after your last word
can measurably change a completion.

### Numbers

**Predict first:** how many tokens is `"1234567"`?

In [ ]:
print("--- numbers get chopped in ways that make arithmetic hard ---")
for t in ["42", "1234567", " 3.14159", "2024", "1999", "1000000"]:
    show(t)

`1234567` becomes `123` + `456` + `7`. The digits are grouped by what was *frequent in training
text*, not by place value, and the grouping shifts depending on the number. A model doing long
addition has to work with chunks that do not line up between operands — which is a large part of why
LLMs are unreliable at arithmetic while being fine at algebra.

### Morphology and typos

In [ ]:
print("--- rare and misspelled words decompose into fragments ---")
for t in ["tokenization", "unhappiness", "antidisestablishmentarianism", "teh", "recieve"]:
    show(t)

Nothing is ever unknown any more. `antidisestablishmentarianism` was almost certainly not a vocabulary
entry, but it is representable as six fragments — and fragments like `establish` and `ism` carry real
meaning the model already knows. That is the payoff over word-level vocabularies.

### The strawberry problem

You have probably seen a model fail to count the letters in `strawberry`. Here is the mechanical
reason.

In [ ]:
word = "strawberry"
pieces = [enc.decode_single_token_bytes(i).decode() for i in enc.encode(word)]

print("what you see            :", list(word))
print("what the model receives :", pieces)
print("r's visible per piece   :", {p: p.count("r") for p in pieces})
print()
show("strawberry")
show(" strawberry")

Two things worth absorbing.

First, the model never sees ten letters. It sees three opaque chunks, and counting `r`s means
recalling the spelling of each chunk from memory rather than looking at it. Character-level questions
are genuinely hard from inside this representation.

Second — and this is the detail people miss — `"strawberry"` is 3 tokens but `" strawberry"` with a
leading space is **1 token**. The same word costs different amounts and decomposes differently
depending on the character before it.

### The token tax: not all languages cost the same

In [ ]:
samples = {
    "English": "Large language models read integers, not text.",
    "Spanish": "Hola, \u00bfc\u00f3mo est\u00e1s?",
    "Hindi": "\u0928\u092e\u0938\u094d\u0924\u0947 \u0926\u0941\u0928\u093f\u092f\u093e",
    "Emoji": "\U0001f600\U0001f680",
    "Python": "  def foo():\n        return 1\n",
}
encodings = ["r50k_base", "cl100k_base", "o200k_base"]
encs = {n: tiktoken.get_encoding(n) for n in encodings}

print(f"{'sample':<9}{'chars':>6}" + "".join(f"{n:>13}" for n in encodings))
print(f"{'':<9}{'':>6}" + "".join(f"{m:>13}" for m in ["GPT-2/3", "GPT-3.5/4", "GPT-4o"]))
print("-" * 54)
for label, s in samples.items():
    counts = "".join(f"{len(encs[n].encode(s)):>13}" for n in encodings)
    print(f"{label:<9}{len(s):>6}{counts}")
print("-" * 54)
for n in encodings:
    print(f"{n:<14} vocab = {encs[n].n_vocab:>7,}")

Read that table by row and by column.

**By row:** 13 characters of Hindi cost 23 tokens under GPT-2's tokenizer while 46 characters of
English cost 9. Since you are billed per token and the context window is measured in tokens, the same
meaning costs several times more in Hindi than in English. This is a real and much-discussed fairness
issue, not a curiosity.

**By column:** as vocabularies grew from 50k to 200k, the non-English cost collapsed — Hindi went
from 23 tokens to 5. That improvement is most of why newer models feel so much better at other
languages. English barely changed, because it was already efficient.

### Tokens are bytes, not characters

One more layer of truth. A token is a sequence of *bytes*, and a single character can be split
mid-encoding.

In [ ]:
e = encs["cl100k_base"]
smiley = "\U0001f600"
ids = e.encode(smiley)
raw = [e.decode_single_token_bytes(i) for i in ids]

print(f"{smiley} is {len(smiley.encode())} UTF-8 bytes and {len(ids)} tokens")
for i, b in zip(ids, raw):
    try:
        shown = b.decode()
    except UnicodeDecodeError:
        shown = "<incomplete UTF-8 -- meaningless alone>"
    print(f"  {i:>7}  {b!r:<10}  {shown}")

print("\nonly reassembled do the bytes mean anything:", b"".join(raw).decode())

This is why streaming APIs sometimes emit a broken character and then fix it: the first token of an
emoji is not a valid character on its own. It also explains why BPE never needs an `<UNK>` token —
at worst it falls back to individual bytes, and *every* possible input is a sequence of bytes.

### Practical calibration

The rule of thumb worth memorising: for ordinary English, **1 token is about 0.75 words**, or
1.3 tokens per word. Clean prose like the paragraph below comes in a little under that; text dense
with punctuation, names, code or numbers runs well over it, so 1.3 is the safer planning figure.

In [ ]:
paragraph = (
    "Large language models do not read text. They read integers. "
    "Every character you type is chopped into tokens, each token is looked up "
    "in a giant table of vectors, and only then does any maths happen."
)
n_words = len(paragraph.split())

for n in encodings:
    t = len(encs[n].encode(paragraph))
    print(f"{n:<14}{t:>4} tokens for {n_words} words  ->  {t/n_words:.2f} tokens/word")

ctx = 128_000
print(f"\nso a {ctx:,}-token context window holds roughly {int(ctx/1.3):,} English words,")
print(f"about {int(ctx/1.3/450):,} pages of a paperback -- but far less Hindi.")

---
## 3. Implement byte-pair encoding yourself (10 min)

`tiktoken` is still a black box: we have watched *what* it does but not *how*. The algorithm is small
enough to write from memory, so let's write it.

BPE training is three steps in a loop:

1. Start with every word split into its individual characters.
2. Count every adjacent pair of symbols across the corpus. Merge the most frequent pair everywhere,
   creating one new symbol.
3. Repeat for a fixed number of merges.

The `</w>` marker below flags the end of a word, so that `est` at the end of `lowest` is a different
symbol from `est` in the middle of a word.

In [ ]:
from collections import Counter

corpus = "low lower lowest newer newest wider widest new low low newer".split()
word_freq = Counter(corpus)

# each word becomes a tuple of symbols; initially those symbols are single characters
state = {tuple(w) + ("</w>",): count for w, count in word_freq.items()}

print("starting point -- frequency, then symbols:")
for symbols, freq in state.items():
    print(f"{freq:>3}  {' '.join(symbols)}")

In [ ]:
def pair_counts(state):
    # count every adjacent symbol pair, weighted by how often its word appears
    counts = Counter()
    for symbols, freq in state.items():
        for a, b in zip(symbols, symbols[1:]):
            counts[(a, b)] += freq
    return counts


def apply_merge(state, pair):
    # replace every adjacent occurrence of `pair` with the single joined symbol
    a, b = pair
    out = {}
    for symbols, freq in state.items():
        merged, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                merged.append(a + b)
                i += 2
            else:
                merged.append(symbols[i])
                i += 1
        out[tuple(merged)] = freq
    return out


print("top pairs before any merging:")
for pair, freq in pair_counts(state).most_common(5):
    print(f"  {pair[0]!r} + {pair[1]!r}  seen {freq}x")

Now the training loop. Every line of output is one learned merge rule — and that ordered list of
rules *is* the trained tokenizer.

In [ ]:
merges = []
for step in range(10):
    counts = pair_counts(state)
    if not counts:
        break
    best, freq = counts.most_common(1)[0]
    merges.append(best)
    state = apply_merge(state, best)
    print(f"merge {step+1:>2}:  {best[0]!r:>8} + {best[1]!r:<8} -> {best[0]+best[1]!r:<10} (seen {freq}x)")

In [ ]:
print("how the corpus is tokenized now:")
for symbols, freq in state.items():
    print(f"{freq:>3}  {' | '.join(symbols)}")

Look at what emerged without anyone specifying it: `low`, `new`, `est</w>` and `er</w>` became single
symbols. The algorithm discovered stems and suffixes purely from co-occurrence counts. Nobody taught
it English morphology.

Now the part that matters — applying the learned rules to a word the tokenizer has **never seen**.

In [ ]:
def bpe_encode(word, merges):
    # tokenize a word by replaying the learned merges in the order they were learned
    state = {tuple(word) + ("</w>",): 1}
    for pair in merges:
        state = apply_merge(state, pair)
    return list(next(iter(state)))


for w in ["lowest", "newer", "slowest", "lowing", "newborn"]:
    seen = "seen in training" if w in word_freq else "NEVER SEEN"
    print(f"{w:<9} -> {str(bpe_encode(w, merges)):<45} ({seen})")

`slowest` never appeared in the training corpus, yet it comes out as `s` + `low` + `est</w>` — the
stem and suffix are recognised, and only the unfamiliar `s` is left stranded. That is generalisation,
and it is the entire reason subword tokenization won.

### How this differs from the real thing

| Our version | Production BPE |
| ----------- | -------------- |
| Symbols start as characters | Symbols start as the **256 possible bytes**, so no input is ever unrepresentable |
| 10 merges | ~100,000 merges (`cl100k_base` = 100,277 tokens) |
| 12 words of corpus | Trillions of characters of web text |
| Words pre-split on spaces | A regex splits text first, deliberately attaching spaces to word starts |
| Pure Python, slow | Rust, and the merges are pre-baked into a lookup table |

Vocabulary size is the interesting hyperparameter, and it is a genuine trade-off.

In [ ]:
for n in encodings:
    e = encs[n]
    t = len(e.encode(paragraph))
    table_params = e.n_vocab * 768
    print(f"{n:<14} vocab {e.n_vocab:>7,}  ->  {t:>3} tokens for our paragraph,"
          f"  {table_params/1e6:>5.1f}M params just to embed the vocabulary")

A bigger vocabulary means fewer tokens per document (cheaper inference, more text per context window)
but a larger embedding table to store and learn. Which brings us to that table.

---
## 4. From token IDs to vectors: the embedding matrix (5 min)

We now have integers. Integers are still not usable — `token 5000` is not "twice" `token 2500`. The
final step is to give each token ID its own learned vector, stored as a row in a matrix
`E` of shape `(vocab_size, d_model)`.

The key insight is that **looking up a row is the same operation as multiplying by a one-hot
vector**, which is exactly the encoding we discarded in segment 1.

In [ ]:
V_demo, d_demo = 8, 4
E = np.random.randn(V_demo, d_demo) * 0.1

print("embedding matrix E, shape", E.shape, "= (vocab_size, d_model)")
print(E)

idx = 5
oh = np.zeros(V_demo)
oh[idx] = 1.0

print(f"\nfor token id {idx}:")
print("  one_hot @ E =", oh @ E)
print("  E[idx]      =", E[idx])
print("  identical?  ", np.allclose(oh @ E, E[idx]))

So one-hot encoding never really went away — it is the *mathematical* definition of the embedding
layer. It is simply never computed that way, because multiplying by a 100,277-wide vector of zeros to
select one row would be absurd. Frameworks call `E[idx]` instead, and PyTorch's `nn.Embedding` is
precisely this: a matrix plus an indexing operation.

Let's do it with a real vocabulary and real token IDs.

In [ ]:
d = 8
table = (np.random.randn(enc.n_vocab, d) * 0.02).astype(np.float32)
print(f"embedding table for the full cl100k vocabulary: {table.shape} "
      f"= {table.nbytes/1e6:.1f} MB at {d} dims")

sentence = "the cat sat on the mat"
ids = enc.encode(sentence)
vectors = table[ids]

print(f"\n{sentence!r}")
print("token ids :", ids)
print("vectors   :", vectors.shape, "= (sequence_length, d_model)")
print(vectors[:3])

That shape, `(sequence_length, d_model)`, is what actually enters a transformer. Everything upstream
of it is bookkeeping.

### What the table costs

**Predict first:** GPT-2 small has 124M parameters in total. What fraction do you think is spent
just on the embedding table?

In [ ]:
for name, V_ in [("r50k_base", 50_257), ("cl100k_base", 100_277), ("o200k_base", 200_019)]:
    for d_ in (768, 4096):
        print(f"{name:<14} d_model={d_:>5}  ->  {V_*d_/1e6:>7.1f}M parameters in the embedding table")
    print()

gpt2_embed = 50_257 * 768
print(f"GPT-2 small: {gpt2_embed/1e6:.1f}M of its 124M parameters are the embedding table "
      f"-- {100*gpt2_embed/124e6:.0f}%")

Roughly a third of GPT-2 small is the vocabulary lookup table. This is why vocabulary size is not a
free parameter, and why small models sometimes **tie** the input embedding and output projection
matrices to avoid paying for it twice.

### The part we have not done yet

Everything in `table` above is **random noise**. Random vectors carry no more meaning than one-hot
vectors did — `cosine(cat, dog)` on a random table is arbitrary.

What makes embeddings useful is that those rows are *learned*, so that geometry ends up encoding
meaning. Segment 5 does exactly that learning, from scratch.

---
## 5. Word2Vec from scratch (15 min)

This is the heart of the hour, and the part
[The Illustrated Word2Vec](https://jalammar.github.io/illustrated-word2vec/) covers. Read it
alongside this segment — we are building the model it describes.

The problem: we want embedding rows whose geometry encodes meaning, but nobody has labelled the
meaning of every word. Where does the training signal come from?

The answer is the **distributional hypothesis**:

> "You shall know a word by the company it keeps." — J.R. Firth, 1957

Words that appear in similar contexts tend to mean similar things. `coffee` and `tea` both get
*drunk*, *brewed*, and taken *hot*. That is a training signal you can extract from raw text with no
annotation at all — **self-supervision**. We invent a fake prediction task, and the useful part is
not the predictions but the weights we learn on the way.

There are two ways to set up the fake task:

```
CBOW: predict the centre word from its context      skip-gram: predict the context from the centre

   the  ___  sat  on                                      the  cat  sat  on
    \    ^    /                                            ^    |    ^
     \   |   /                                             |    |    |
      context -> "cat"                                    "cat" predicts each neighbour
```

We use **skip-gram**, which works better on small corpora and is what the article focuses on.

### The corpus

An honest warning up front: real Word2Vec needs billions of words. To get visible structure in a few
seconds, we use a deliberately synthetic corpus — each "sentence" is a concept word together with its
attributes, shuffled.

This is a cheat, but an instructive one. It makes the co-occurrence statistics that Word2Vec relies
on unusually clean, so the geometry that normally takes hours to emerge appears in seconds. The
algorithm below is exactly the real one.

In [ ]:
CONCEPTS = {
    "king":     ["royal", "ruler", "male", "adult"],
    "queen":    ["royal", "ruler", "female", "adult"],
    "prince":   ["royal", "heir", "male", "child"],
    "princess": ["royal", "heir", "female", "child"],
    "man":      ["commoner", "person", "male", "adult"],
    "woman":    ["commoner", "person", "female", "adult"],
    "boy":      ["commoner", "person", "male", "child"],
    "girl":     ["commoner", "person", "female", "child"],
    "actor":    ["performer", "person", "male", "adult"],
    "actress":  ["performer", "person", "female", "adult"],
}

rng = np.random.default_rng(0)
sentences = []
for _ in range(40):                       # 40 passes over the concept set
    for word, attributes in CONCEPTS.items():
        s = [word] + list(attributes)
        rng.shuffle(s)                    # shuffle so co-occurrence is order-independent
        sentences.append(s)
rng.shuffle(sentences)

print(f"{len(sentences)} sentences, {sum(len(s) for s in sentences)} word occurrences")
print("\nfirst five:")
for s in sentences[:5]:
    print("  ", " ".join(s))

In [ ]:
from collections import Counter

counts = Counter(w for s in sentences for w in s)
vocab_words = [w for w, _ in counts.most_common()]
w2i = {w: i for i, w in enumerate(vocab_words)}
Vw = len(vocab_words)

print(f"vocabulary: {Vw} words")
print(dict(counts.most_common()))

### Sliding a window to make training pairs

Skip-gram turns raw text into supervised `(centre, context)` pairs by sliding a window over every
sentence. That is all the "labelling" there is.

In [ ]:
WINDOW = 4

pairs = []
for s in sentences:
    ids = [w2i[w] for w in s]
    for i, centre in enumerate(ids):
        lo, hi = max(0, i - WINDOW), min(len(ids), i + WINDOW + 1)
        for j in range(lo, hi):
            if j != i:
                pairs.append((centre, ids[j]))
pairs = np.array(pairs, dtype=np.int64)

print(f"{len(pairs):,} training pairs from {len(sentences)} sentences")
print("\nthe first sentence:", " ".join(sentences[0]))
print("produces pairs:")
for c, o in pairs[:8]:
    print(f"   centre={vocab_words[c]:<10} context={vocab_words[o]}")

### Why the obvious loss is unaffordable

The textbook formulation predicts the context word with a softmax over the whole vocabulary. That
means computing a score for **every word in the vocabulary** on **every training pair**.

In [ ]:
V_real, k_neg = 100_277, 5
print(f"full softmax:      {V_real:>7,} dot products per training pair")
print(f"negative sampling: {k_neg + 1:>7,} dot products per training pair")
print(f"speedup:           {V_real/(k_neg+1):>7,.0f}x")

So Word2Vec replaces the question entirely. Instead of

> *"given `king`, which of my 100,277 words comes next?"* — a hard multi-class problem,

it asks

> *"given `king` and `royal`, is this a real pair from the corpus, or noise I invented?"* — an easy
> binary problem.

For each real pair we draw `k` **negative samples**: random words that probably do not belong. The
model learns to score real pairs high and fake pairs low. This is **negative sampling**, and it is
the single trick that made Word2Vec fast enough to run on billions of words in 2013.

Negatives are not drawn uniformly. They come from the unigram frequency raised to the power 0.75, an
empirical choice that pulls rare words up and common words down.

In [ ]:
freq = np.array([counts[w] for w in vocab_words], dtype=float)
noise_dist = freq ** 0.75
noise_dist /= noise_dist.sum()
plain = freq / freq.sum()

print(f"{'word':<10}{'count':>7}{'p (raw)':>10}{'p (^0.75)':>11}   effect")
for i in list(np.argsort(-freq))[:4] + list(np.argsort(freq))[:3]:
    w = vocab_words[i]
    effect = "boosted" if noise_dist[i] > plain[i] else "suppressed"
    print(f"{w:<10}{int(freq[i]):>7}{plain[i]:>10.3f}{noise_dist[i]:>11.3f}   {effect}")

Here is one training example in full, exactly like the tables in the article: one positive row
labelled 1, and `k` sampled negatives labelled 0.

In [ ]:
demo_centre, demo_context = pairs[0]
demo_negs = rng.choice(Vw, size=k_neg, p=noise_dist)

print(f"{'centre':<10}{'candidate':<12}{'label':>6}   meaning")
print(f"{vocab_words[demo_centre]:<10}{vocab_words[demo_context]:<12}{1:>6}   real pair from the corpus")
for n in demo_negs:
    print(f"{vocab_words[demo_centre]:<10}{vocab_words[n]:<12}{0:>6}   sampled noise")

### The model

Two matrices, both `(vocab_size, dim)`:

- `W_in` — the vector for a word **when it is the centre word**. These are the embeddings we keep.
- `W_out` — the vector for a word **when it is a context word**. Scaffolding, thrown away at the end.

Why two? Because a word should not be predicting itself with a high score just because
`v · v` is large. Separate roles keep the objective well behaved. Real Word2Vec does the same and
discards `W_out` too.

For a centre vector `v` and candidate vector `u`, the model predicts

```
p(real) = sigmoid(u · v)
```

and the gradient of the binary cross-entropy loss is beautifully simple: `(prediction - label)`.
That single expression drives both updates below.

In [ ]:
DIM, EPOCHS, LR, NEG = 32, 15, 0.01, 5

W_in = rng.normal(0, 0.1, (Vw, DIM))     # the embeddings we care about
W_out = np.zeros((Vw, DIM))              # context vectors, discarded later

history = []
for epoch in range(EPOCHS):
    order = rng.permutation(len(pairs))
    negs = rng.choice(Vw, size=(len(pairs), NEG), p=noise_dist)
    alpha = LR * (1 - epoch / EPOCHS) + LR * 0.01     # linear learning-rate decay
    total = 0.0

    for n, p in enumerate(order):
        centre, context = pairs[p]
        targets = np.concatenate(([context], negs[n]))   # 1 positive + NEG negatives
        labels = np.zeros(NEG + 1)
        labels[0] = 1.0

        v = W_in[centre].copy()
        u = W_out[targets]

        pred = 1.0 / (1.0 + np.exp(-np.clip(u @ v, -20, 20)))       # sigmoid scores
        total += -(np.log(pred[0] + 1e-10) + np.log(1 - pred[1:] + 1e-10).sum())

        grad = pred - labels                 # the entire derivative
        W_in[centre] -= alpha * (grad @ u)
        W_out[targets] -= alpha * np.outer(grad, v)

    history.append(total / len(pairs))
    print(f"epoch {epoch+1:>2}/{EPOCHS}   loss {history[-1]:.4f}")

The loss drops steeply and then flattens around 2.2 rather than heading for zero. That is expected
and not a bug: some negative samples genuinely *are* plausible context words, so the model is
penalised for correctly thinking so. The floor is the noise in the task itself.

### Did it learn anything?

The test is whether cosine similarity in `W_in` now tracks meaning — the thing one-hot encoding
could not do at all.

In [ ]:
def unit(M):
    return M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12)


E_learned = unit(W_in)


def nearest(word, k=4):
    sims = E_learned @ E_learned[w2i[word]]
    ranked = [(vocab_words[i], float(sims[i])) for i in np.argsort(-sims) if vocab_words[i] != word]
    return ranked[:k]


for w in ["king", "queen", "boy", "royal"]:
    hits = ", ".join(f"{x} ({s:.2f})" for x, s in nearest(w))
    print(f"nearest to {w:<8} {hits}")

Compare that to segment 1, where every similarity was exactly 0.00. The numbers now mean something:
`king` leads with `queen` and `ruler`, `boy` leads with `girl`, and the ranking is stable rather
than arbitrary. The vocabulary has sorted itself into royalty, commoners, ages and genders without
anyone supplying those categories.

### Analogies: the famous part

If the gender difference is stored as a consistent *direction* in the space, then subtracting and
adding along it should work like arithmetic. `king - man + woman` should land near `queen`.

In [ ]:
def analogy(a, b, c, k=3):
    target = E_learned[w2i[a]] - E_learned[w2i[b]] + E_learned[w2i[c]]
    target /= np.linalg.norm(target)
    sims = E_learned @ target
    ranked = [(vocab_words[i], float(sims[i])) for i in np.argsort(-sims) if vocab_words[i] not in (a, b, c)]
    return ranked[:k]


for a, b, c in [("king", "man", "woman"), ("prince", "boy", "girl"),
                ("actor", "man", "woman"), ("king", "male", "female"),
                ("king", "queen", "princess")]:
    hits = ", ".join(f"{x} ({s:.2f})" for x, s in analogy(a, b, c))
    print(f"{a} - {b} + {c:<8} = {hits}")

Nobody told the model about gender, royalty or age. It only ever saw which words appeared near which
other words, and the structure fell out of the counting.

That is the result that made Word2Vec famous in 2013, and it is the moment worth sitting with: the
*offset* between `man` and `woman` is roughly the same vector as the offset between `king` and
`queen`, so the space has axes that correspond to human concepts even though no human labelled them.

### Seeing the vectors

The article visualises embeddings as coloured bars. Same idea here — each row is one word's vector,
each column one dimension.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

show = ["king", "queen", "prince", "princess", "man", "woman", "boy", "girl", "actor", "actress"]
M = E_learned[[w2i[w] for w in show]]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

im0 = axes[0].imshow(M, cmap="RdBu", vmin=-0.4, vmax=0.4, aspect="auto")
axes[0].set_yticks(range(len(show)))
axes[0].set_yticklabels(show)
axes[0].set_xlabel("embedding dimension")
axes[0].set_title("learned vectors: one row per word")
fig.colorbar(im0, ax=axes[0])

S = M @ M.T
im1 = axes[1].imshow(S, cmap="viridis", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(show)))
axes[1].set_xticklabels(show, rotation=90)
axes[1].set_yticks(range(len(show)))
axes[1].set_yticklabels(show)
axes[1].set_title("cosine similarity (compare with segment 1)")
fig.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

Look for the column stripes in the left panel: dimensions where all the male words share a colour and
all the female words share the opposite one. That is the gender direction the analogy arithmetic
exploited.

The right panel is the payoff. In segment 1 this matrix was the identity — a diagonal of ones and
nothing else. Now it has visible blocks: royals resemble royals, children resemble children, and the
diagonal is no longer the only signal.

### What real Word2Vec adds

| Our version | Mikolov et al. 2013 |
| ----------- | ------------------- |
| 400 synthetic sentences | billions of words of news and Wikipedia |
| 32 dimensions | 300 dimensions |
| 5 negative samples | 5-20 (more for small datasets) |
| every pair used | frequent words like `the` subsampled away |
| fixed window of 4 | window size sampled randomly per pair, so near words count more |
| plain SGD in Python | multithreaded C, hours on one machine |

Set expectations honestly: on a real corpus, analogies work maybe 50-70% of the time on standard
benchmarks, not the near-100% our engineered corpus produces. Word2Vec is a real but noisy effect.

---
## 6. Why Word2Vec is not enough (7 min)

Word2Vec was the state of the art in 2013 and is now a stepping stone. Two limitations killed it, and
understanding both tells you what a transformer is actually for.

### Limitation 1: one vector per word, forever

Our `W_in` has exactly one row per word. So a word with two meanings gets one vector — the average of
its senses, useful for neither.

In [ ]:
s1 = "i sat on the river bank and watched the water"
s2 = "i deposited the cheque at the bank this morning"

bank_id = enc.encode(" bank")[0]

print(f"{s1!r}\n  contains token {bank_id}?  {bank_id in enc.encode(s1)}")
print(f"{s2!r}\n  contains token {bank_id}?  {bank_id in enc.encode(s2)}")
print(f"\nboth sentences use the very same id for ' bank': {bank_id}")
print("so the static embedding table hands both of them the identical row:")
print("  ", table[bank_id][:5], "...")

The river and the money get the same vector. This is what **contextual embeddings** fix: a
transformer starts from that same static row, then uses attention to let every token rewrite itself
based on its neighbours. After a few layers, `bank` near `river` and `bank` near `cheque` have
genuinely different vectors.

The static table never disappears though — it is still layer zero of every LLM. It is the *starting
point* that context then modifies.

### Limitation 2: no sense of order

Word2Vec is a bag-of-words model. Embeddings alone carry no position, so word order is invisible.

In [ ]:
a = "the dog bites the man"
b = "the man bites the dog"

ids_a, ids_b = enc.encode(a), enc.encode(b)
print(f"{a!r} -> {ids_a}")
print(f"{b!r} -> {ids_b}")
print("same multiset of tokens? ", sorted(ids_a) == sorted(ids_b))

bag_a = table[ids_a].sum(axis=0)
bag_b = table[ids_b].sum(axis=0)
print("\nsum of embedding vectors identical?", np.allclose(bag_a, bag_b))
print("cosine between the two 'sentence vectors':", round(cosine(bag_a, bag_b), 6))

Two sentences with opposite meanings, one identical representation. Something has to inject position,
which is what **positional encodings** do: before the first transformer layer, a position-dependent
vector is added to each token embedding, so the same word at position 2 and position 7 no longer
starts out identical.

### Where this leads

The full modern pipeline, with today's hour marked:

```
text -> [BPE tokenizer] -> ids -> [embedding table] -> vectors -> [+ positional encoding]
        ^ segments 2-3            ^ segments 4-5                  ^ tomorrow
     -> [attention layers] -> contextual vectors -> [output projection] -> next-token probabilities
        ^ tomorrow
```

Everything left of the arrow marked *tomorrow* is what you built today. Two related ideas worth
knowing exist just off this diagram:

- **Sentence embeddings.** One vector for a whole passage rather than per token, produced by models
  like `sentence-transformers`. This is what vector databases store.
- **Retrieval (RAG).** Embed your documents, embed the question, and use cosine similarity — the same
  function from segment 1 — to find relevant passages. The plumbing you learned today *is* the
  plumbing of semantic search.

### The hour in one paragraph

Text is chopped into subword tokens by a merge table learned from data, each token is an integer,
each integer indexes a row of a learned matrix, and those rows arrange themselves so that geometric
relationships encode semantic ones. Everything after that is the model.

---
## 7. Exercises

Six exercises, roughly in notebook order. Work in the scratch cell below before scrolling on — the
solutions are runnable cells further down, so they are easy to spoil.

**Exercise 1 — the cost of a space.** For a list of words, compare `len(enc.encode(w))` with
`len(enc.encode(" " + w))`. How many get *cheaper* with a leading space? Find at least one string
where the space makes it more expensive, and explain what class of string behaves that way.

**Exercise 2 — which years are one token?** *Predict first:* of the 151 years from 1900 to 2050,
how many does `cl100k_base` encode as a single token? Check, look at the actual pieces of `"2024"`,
and work out the rule being applied. Does `o200k_base` do any better?

**Exercise 3 — what does each merge buy you?** Wrap segment 3's loop into
`train_bpe(corpus, num_merges)` and record the average number of symbols per word after every
merge. Plot that curve over 40 merges on a larger corpus. Where does it flatten, and what does that
say about choosing a vocabulary size?

**Exercise 4 — the backward pass of an embedding lookup.** Forward is `vectors = E[ids]`. Given an
upstream gradient `grad_vectors` of the same shape, compute `grad_E`. Use an `ids` list with a
repeated token, verify the result against the one-hot version `onehot.T @ grad_vectors`, and show
what goes wrong if you write `grad_E[ids] += grad_vectors`.

**Exercise 5 — generate the training pairs.** Write `skipgram_pairs(tokens, window)` returning
every `(centre, context)` pair with the window truncated at the ends of the sequence. Count the
pairs for windows 1, 2 and 5, and check the totals against the closed form `2*w*n - w*(w+1)`. Work
out for which windows that formula can possibly be right.

**Exercise 6 — inside negative sampling (stretch).**
(a) Build the noise distribution `p(w) ∝ count(w)**0.75` for a toy frequency table and show what
the 0.75 exponent does relative to raw frequency.
(b) For one `(centre, context, negatives)` triple, write the negative-sampling loss and its
analytic gradients, then check them against finite differences.

In [ ]:
# Everything the exercises need, restated so this section stands alone.
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
enc_o200k = tiktoken.get_encoding("o200k_base")


def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -20.0, 20.0)))


def pieces_of(text, e=None):
    e = e or enc
    return [e.decode_single_token_bytes(i).decode("utf-8", "replace") for i in e.encode(text)]


print("exercise setup ready")

In [ ]:
# Scratch space for the exercises. Try them here first.

---
## Solutions

Spoilers from here down. Each solution is a runnable cell, so you can also treat them as worked
examples to modify.

### Solution 1 — the cost of a space

Words common enough to have their own token are one token either way, so most of the list does not
move. The gap opens on longer or rarer words: BPE learned its merges on prose where a word almost
always follows a space, so for those the space-prefixed form *is* the canonical unit and the bare
form is the odd one that has to be assembled from fragments — `strawberry` is the extreme case at
three tokens against one. Numbers go the other way, and that is the class worth remembering: a
space never merges into a digit run, so `" 42"` pays for the space as its own token.

In [ ]:
ex1_words = ["cat", "strawberry", "tokenization", "hello", "queen", "Python",
             "internationalization", "42", "OpenAI", "unhappiness"]

print(f"{'string':<22}{'bare':>6}{'+space':>8}   pieces with the leading space")
cheaper = worse = 0
for w in ex1_words:
    bare, spaced = len(enc.encode(w)), len(enc.encode(" " + w))
    cheaper += spaced < bare
    worse += spaced > bare
    print(f"{w:<22}{bare:>6}{spaced:>8}   {pieces_of(' ' + w)}")

print(f"\n{cheaper}/{len(ex1_words)} got cheaper with a leading space, {worse} got more expensive.")
print("the expensive one is the number: a space never merges into digits.")

### Solution 2 — which years are one token?

None of them — not one year from 1900 to 2050 is a single token, in either encoding. The reason is
not the merge table but the step *before* it: both tokenizers pre-split text with a regex that caps
a digit run at three characters (`\p{N}{1,3}`), so BPE never even sees `2024` as a candidate to
merge. Every four-digit year arrives as `202` + `4`, and `" 2024"` is three tokens because the
space cannot join the digits either.

That cap is deliberate. Without it the vocabulary would fill with thousands of arbitrary numeric
strings. The cost is that place value is invisible: `1234567` splits `123|456|7` while `12345`
splits `123|45`, so the same digit sits in a different chunk depending on the number's length. Two
operands in an addition problem therefore do not line up at all.

In [ ]:
for name, e in [("cl100k_base", enc), ("o200k_base", enc_o200k)]:
    single = [y for y in range(1900, 2051) if len(e.encode(str(y))) == 1]
    print(f"{name:<12} single-token years in 1900-2050: {len(single)} of 151")

print("\nhow numbers actually split (cl100k_base):")
for s in ["2024", " 2024", "12345", "1234567", "3.14159", "007"]:
    p = pieces_of(s)
    print(f"  {s!r:<10} {len(p)} tokens  {p}")

print("\nsame strings under o200k_base:")
for s in ["2024", "1234567"]:
    print(f"  {s!r:<10} {pieces_of(s, enc_o200k)}")

### Solution 3 — what does each merge buy you?

The curve drops steeply and then flattens: the first handful of merges catch the pairs that occur
everywhere (`o`+`w`, then `l`+`ow`, then `e`+`r`), and each later merge applies to fewer and fewer
positions — the last few merges each fix up a single word. That is
the shape behind every vocabulary-size decision, and segment 2's table is the same curve at
production scale: growing the vocabulary from 50k to 200k cut Hindi from 23 tokens to 5, but left
English sitting at 9, because the first 50k merges already covered English. The extra 150k rows
were bought almost entirely for other languages and for code.

In [ ]:
def avg_symbols_per_word(state):
    return sum(len(sym) * f for sym, f in state.items()) / sum(state.values())


def train_bpe(corpus, num_merges):
    state = {tuple(w) + ("</w>",): c for w, c in Counter(corpus).items()}
    merges, curve = [], [avg_symbols_per_word(state)]
    for _ in range(num_merges):
        counts = pair_counts(state)
        if not counts:
            break
        best, _freq = counts.most_common(1)[0]
        merges.append(best)
        state = apply_merge(state, best)
        curve.append(avg_symbols_per_word(state))
    return merges, curve


bigger_corpus = ("low low low lower lowest slow slower slowest new new newer newest news "
                 "wide wider widest widely narrow narrower narrowest "
                 "quick quicker quickest quickly slowly lowly newly").split()

ex3_merges, curve = train_bpe(bigger_corpus, 40)
print(f"{len(ex3_merges)} merges learned on {len(set(bigger_corpus))} distinct words")
print(f"average symbols per word: {curve[0]:.2f} -> {curve[-1]:.2f}")
print("first ten merges:", ["".join(m) for m in ex3_merges[:10]])

plt.figure(figsize=(6.5, 3.4))
plt.plot(range(len(curve)), curve, marker=".", linewidth=1.5)
plt.xlabel("merges learned  (= vocabulary size)")
plt.ylabel("avg symbols per word")
plt.title("BPE compression has sharply diminishing returns")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Solution 4 — the backward pass of an embedding lookup

Forward, the lookup selects rows. Backward, it does the opposite: it **scatters** each row of the
upstream gradient back to the row of `E` it came from, and when a token appears more than once in
the sequence its gradients must *accumulate*. This is exactly what `onehot.T @ grad_vectors`
computes, which is the useful sanity check — the one-hot view tells you what the answer must be.

The trap is that `grad_E[ids] += grad_vectors` looks right and is wrong. NumPy's fancy indexing
does not accumulate on duplicate indices; it buffers and writes once, so a token appearing three
times gets only the last of its three gradients. `np.add.at` (or `scatter_add_` in PyTorch) is the
version that accumulates. Common tokens like `the` repeat constantly, so this bug quietly wrecks
exactly the rows that matter most.

In [ ]:
rng = np.random.default_rng(0)
V_ex, d_ex = 10, 4
E_ex = rng.normal(size=(V_ex, d_ex)) * 0.1

ids_ex = [3, 7, 3, 1, 3]                 # token 3 appears three times
vectors = E_ex[ids_ex]                   # forward: gather
grad_vectors = rng.normal(size=vectors.shape)

grad_E = np.zeros_like(E_ex)
np.add.at(grad_E, ids_ex, grad_vectors)  # backward: scatter-add

onehot = np.zeros((len(ids_ex), V_ex))
onehot[np.arange(len(ids_ex)), ids_ex] = 1.0
grad_E_onehot = onehot.T @ grad_vectors

print("scatter-add matches the one-hot matmul:", np.allclose(grad_E, grad_E_onehot))
print("rows that received gradient:", sorted(set(ids_ex)), "- the other 7 stay exactly zero")

buggy = np.zeros_like(E_ex)
buggy[ids_ex] += grad_vectors            # the classic bug
print("\nnaive += agrees:", np.allclose(buggy, grad_E))
print("row 3, correct (sum of 3 gradients):", grad_E[3])
print("row 3, naive   (only the last one) :", buggy[3])
print("row 3, last gradient alone         :", grad_vectors[4])

### Solution 5 — generate the training pairs

Note what the window does at the edges: it is truncated, not wrapped and not padded, so the first
and last tokens produce fewer pairs than the middle ones. That is the whole reason the count is
`2*w*n - w*(w+1)` rather than `2*w*n` — the correction term counts the pairs lost off the two ends.
It holds for any `w <= n - 1`; past that the window already spans the whole sequence, the count
saturates at `n*(n-1)`, and the formula starts subtracting edges that were never there.

Window size is a real modelling choice rather than a tuning detail. Small windows (1–2) capture
syntax, since only immediate neighbours count, and tend to group words that are interchangeable in
a sentence. Large windows (5–10) capture topic, grouping words that show up in the same kind of
document. Real Word2Vec also samples the window size per centre word, which weights near
neighbours more heavily without any extra machinery.

In [ ]:
def skipgram_pairs(tokens, window):
    pairs = []
    for i, centre in enumerate(tokens):
        lo, hi = max(0, i - window), min(len(tokens), i + window + 1)
        for j in range(lo, hi):
            if j != i:
                pairs.append((centre, tokens[j]))
    return pairs


ex5_tokens = "the quick brown fox jumps over the lazy dog".split()
ex5_n = len(ex5_tokens)

for w in (1, 2, 5):
    ex5_pairs = skipgram_pairs(ex5_tokens, w)
    formula = 2 * w * ex5_n - w * (w + 1)
    print(f"window={w}: {len(ex5_pairs):>3} pairs  ({len(ex5_pairs)/ex5_n:4.1f} per centre word)  "
          f"formula says {formula}  match={len(ex5_pairs) == formula}")

w_big = 12
print(f"\nwindow={w_big} (bigger than the sequence): {len(skipgram_pairs(ex5_tokens, w_big))} pairs, "
      f"saturated at n*(n-1) = {ex5_n*(ex5_n-1)}, "
      f"while the formula claims {2*w_big*ex5_n - w_big*(w_big+1)}")

print("\nfirst pairs at window=2:")
for centre, context in skipgram_pairs(ex5_tokens, 2)[:8]:
    print(f"  ({centre:<6} -> {context})")

### Solution 6 — inside negative sampling

**(a)** The `0.75` exponent is a compromise between two bad options. Sampling negatives in
proportion to raw frequency means almost every negative is `the` or `of`, which teaches the model
nothing. Sampling uniformly means negatives are almost always rare junk that was never plausible
context anyway, so the task is too easy. Raising counts to `0.75` flattens the distribution: common
words are still sampled most, but rare words get several times their raw share. The exponent was
found empirically in the original paper, and it stuck.

**(b)** The gradient is the reason negative sampling is so cheap. For each of the `k+1` targets the
error term is just `sigmoid(score) - label`, so the update touches one centre row and `k+1` context
rows per pair. Compare that with a full softmax, which would touch all 100,277 rows for every
single training pair. The finite-difference check below is worth internalising as a habit: any time
you hand-derive a gradient, verify it numerically before you trust a training curve.

In [ ]:
# (a) the noise distribution
labels_a = ["the", "of", "king", "queen", "zebra", "quokka", "xylose"]
counts_a = np.array([1000, 500, 100, 50, 10, 5, 1], dtype=float)

p_raw = counts_a / counts_a.sum()
p_075 = counts_a**0.75 / (counts_a**0.75).sum()
p_unif = np.full_like(counts_a, 1 / len(counts_a))

print(f"{'word':<8}{'count':>7}{'p_raw':>9}{'p^0.75':>9}{'uniform':>9}{'lift':>7}")
for w, c, a, b, u in zip(labels_a, counts_a, p_raw, p_075, p_unif):
    print(f"{w:<8}{int(c):>7}{a:>9.4f}{b:>9.4f}{u:>9.4f}{b/a:>6.2f}x")
print("\nrare words get lifted, common words damped -- but the ordering is preserved.")

In [ ]:
# (b) negative-sampling loss, analytic gradients, and a finite-difference check
rng = np.random.default_rng(1)
d, k = 5, 3
v_c = rng.normal(size=d) * 0.5          # the centre word vector
U = rng.normal(size=(k + 1, d)) * 0.5   # row 0 = the true context word, rows 1..k = negatives
y = np.zeros(k + 1)
y[0] = 1.0                              # labels: real pair 1, noise 0


def loss(v_c, U):
    s = sigmoid(U @ v_c)
    return float(-np.sum(y * np.log(s + 1e-12) + (1 - y) * np.log(1 - s + 1e-12)))


err = sigmoid(U @ v_c) - y              # the entire derivative, in one line
grad_vc = err @ U
grad_U = np.outer(err, v_c)

eps = 1e-6
num_vc = np.zeros_like(v_c)
for i in range(d):
    up, dn = v_c.copy(), v_c.copy()
    up[i] += eps
    dn[i] -= eps
    num_vc[i] = (loss(up, U) - loss(dn, U)) / (2 * eps)

num_U = np.zeros_like(U)
for i in range(k + 1):
    for j in range(d):
        up, dn = U.copy(), U.copy()
        up[i, j] += eps
        dn[i, j] -= eps
        num_U[i, j] = (loss(v_c, up) - loss(v_c, dn)) / (2 * eps)

print(f"loss = {loss(v_c, U):.4f}")
print(f"max |analytic - numeric| for the centre vector : {np.abs(grad_vc - num_vc).max():.2e}")
print(f"max |analytic - numeric| for the context rows  : {np.abs(grad_U - num_U).max():.2e}")
print("\nper-target error (sigmoid(score) - label):")
for i, (e_, lab) in enumerate(zip(err, y)):
    role = "true context" if lab == 1 else f"negative {i}"
    print(f"  {role:<14} score={float((U @ v_c)[i]):>7.3f}  error={e_:>7.3f}")

---
## Self-check quiz

Answer these without scrolling up. If one stumps you, reread the segment named in its answer.

1. Two distinct one-hot vectors have a cosine similarity of exactly 0.0. Why exactly, and why is
   that fatal rather than merely inconvenient?
2. `"strawberry"` is three tokens but `" strawberry"` is one. Why, and what does that imply about
   trailing spaces in a prompt?
3. Why does a byte-level BPE tokenizer never need an `<UNK>` token?
4. What exactly is the artefact produced by training a BPE tokenizer, and how is it used to encode a
   word that never appeared in training?
5. Name one benefit and one cost of doubling the vocabulary size.
6. In what sense is `E[idx]` a matrix multiplication, and why is it never implemented as one?
7. What is the distributional hypothesis, and what supervision does it give you for free?
8. Why is a softmax over the full vocabulary too expensive for Word2Vec, and what does negative
   sampling replace it with?
9. Word2Vec trains two matrices. What is each one for, and which do you keep?
10. Give two things a transformer's embeddings can represent that Word2Vec's cannot, with a
    concrete example of each.

<details>
<summary><b>Answers</b></summary>

1. Because a one-hot vector has its single 1 in a different position for every word, so the dot
    product of two distinct ones has no overlapping non-zero terms and is 0 by construction — not
    approximately, exactly. It is fatal because there is then no notion of similarity at all: the
    model cannot generalise anything it learns about `dog` to `puppy`, and must learn every word
    independently. *(segment 1)*
2. Because the space is part of the token. BPE learned its merges on prose where words follow
    spaces, so `" strawberry"` is a single frequent unit while the bare form has to be assembled
    from `str` + `aw` + `berry`. Practically: a trailing space at the end of your prompt changes the
    tokenization of everything after it and can measurably change the completion. *(segment 2)*
3. Because the symbol alphabet starts as the 256 possible byte values, and every possible input is a
    sequence of bytes. In the worst case a string falls back to individual byte tokens, so there is
    nothing left to be "unknown". *(segments 2–3)*
4. An **ordered list of merge rules** — the vocabulary is just a by-product of it. To encode an
    unseen word you split it into its base symbols and replay every merge rule in the order it was
    learned; whichever merges apply, apply. `slowest` becomes `s` + `low` + `est</w>` even though the
    trainer never saw it. *(segment 3)*
5. Benefit: fewer tokens per document, so cheaper inference, more text per context window, and
    better handling of non-English text. Cost: a linearly larger embedding table (`vocab x d_model`
    parameters) to store and learn, with each row seen less often during training. *(segments 2–4)*
6. `E[idx]` is identical to `one_hot(idx) @ E` — that is the definition of the embedding layer. It is
    never computed that way because multiplying a 100,277-wide vector of zeros by the whole matrix
    to select one row is enormously wasteful when indexing gives the same answer. *(segment 4)*
7. That words appearing in similar contexts tend to have similar meanings. The free supervision is
    that raw unlabelled text already tells you which words co-occur, so you can build
    `(centre, context)` training pairs from any corpus without a single human annotation.
    *(segment 5)*
8. Because the softmax denominator sums over every word in the vocabulary, so each of billions of
    training pairs would touch all 100k+ output rows. Negative sampling reframes the task as binary
    classification — is this pair real or noise? — and uses one true context word plus `k` sampled
    noise words, turning the update into `k+1` dot products. *(segment 5)*
9. An input/centre matrix `W_in` and an output/context matrix `W_out`; every word has a vector in
    each, used depending on the role it is playing in a pair. You keep `W_in` — those are "the"
    word embeddings — and throw `W_out` away, though averaging the two is a known alternative.
    *(segment 5)*
10. **Context:** one vector per word cannot separate senses, so `bank` in "river bank" and "bank
    account" gets the identical Word2Vec vector, while attention gives each occurrence a different
    vector shaped by its neighbours. **Order:** `dog bites man` and `man bites dog` contain the same
    tokens, so any order-blind representation scores them identically; transformers add positional
    information so the two are distinguishable. *(segment 6)*

</details>

---
## Where to go next

Today ended on a limitation with an obvious shape: a lookup table gives each word exactly one
vector, forever, and no amount of extra training data fixes `bank`. The mechanism that does fix it
is **self-attention** — let every token's vector be recomputed as a weighted mixture of the other
tokens in the sequence, with the weights themselves learned from content. That is the natural
Day 02, and it lands directly on top of what you built here: the `(seq_len, d_model)` matrix
produced in segment 4 is precisely the input attention consumes.

Built from scratch, one day of attention covers queries, keys and values, why the scores are scaled
by `sqrt(d_k)`, why multiple heads exist, and the positional encoding that answers the
`dog bites man` problem — ending with a working single-head attention block in numpy.

Two reasonable alternatives if you would rather branch:

- **Sentence embeddings and retrieval / RAG** — the same geometry applied to whole documents, and
  the most directly useful thing to build at work.
- **Training dynamics** — softmax, cross-entropy and backpropagation written out by hand, if
  exercise 6's gradient check felt like the interesting part rather than the fiddly part.